# Trabalho Grau B - Reconhecimento de imagem e transfer learning

## Integrantes

- Arthur Schallenberger
- Giovani de Souza
- Leonardo Fronza
- Renan Milech Pereira

---

**Docente:** Prof. Gabriel de Oliveira Ramos

## 2.1 Descrição do Problema e Dataset

### 2.1.1 Descrição do Problema

O problema abordado neste trabalho é a **classificação multiclasse de imagens de bolas esportivas**. Dado um conjunto de imagens coloridas, o objetivo é treinar um modelo capaz de identificar corretamente a qual modalidade esportiva pertence a bola presente na imagem, dentre 15 categorias distintas.

Trata-se de um problema de **visão computacional supervisionada**, em que cada imagem possui um único rótulo associado (classe da bola). O desafio central está na variação visual entre as classes — algumas bolas possuem formas, texturas e padrões muito distintos (como a bola de futebol americano e a de tênis de mesa), enquanto outras apresentam características visuais próximas (como bola de cricket, hockey e tênis), exigindo que o modelo aprenda representações discriminativas e generalizáveis.

A escolha desse domínio é motivada pela disponibilidade de dados rotulados e pela aplicabilidade prática em sistemas de análise de transmissões esportivas, arbitragem automatizada e catalogação de conteúdo multimídia.

---

### 2.1.2 Dataset

O dataset utilizado é o **Sports Ball Image Recognition**, disponível publicamente na plataforma [Kaggle](https://www.kaggle.com/). Ele é composto por imagens JPEG de bolas de 15 modalidades esportivas diferentes, já organizadas em subpastas por classe e divididas entre conjuntos de treino e teste.

#### Estrutura de diretórios

```
archive/
├── train/
│   ├── american_football/
│   ├── baseball/
│   ├── ...
│   └── volleyball/
└── test/
    ├── american_football/
    ├── baseball/
    ├── ...
    └── volleyball/
```

#### Classes e distribuição de imagens

O dataset contém **15 classes**, com a seguinte distribuição por split:

| Classe               | Treino | Teste | Total |
|----------------------|--------|-------|-------|
| american_football    | 384    | 96    | 480   |
| baseball             | 400    | 100   | 500   |
| basketball           | 340    | 86    | 426   |
| billiard_ball        | 646    | 162   | 808   |
| bowling_ball         | 440    | 111   | 551   |
| cricket_ball         | 581    | 146   | 727   |
| football             | 604    | 151   | 755   |
| golf_ball            | 549    | 138   | 687   |
| hockey_ball          | 530    | 133   | 663   |
| hockey_puck          | 390    | 98    | 488   |
| rugby_ball           | 493    | 124   | 617   |
| shuttlecock          | 429    | 108   | 537   |
| table_tennis_ball    | 620    | 156   | 776   |
| tennis_ball          | 490    | 123   | 613   |
| volleyball           | 432    | 109   | 541   |
| **Total**            | **7.328** | **1.841** | **9.169** |

A divisão treino/teste segue uma proporção aproximada de **80%/20%**, padrão comum em benchmarks de visão computacional.

#### Características das imagens

- **Formato:** JPEG (`.jpg`)
- **Resolução:** variada — as imagens originais possuem dimensões heterogêneas (desde 225×225 até resoluções maiores como 1920×1080 e 2048×1152)
- **Canais:** RGB (3 canais de cor)
- **Pré-processamento necessário:** todas as imagens serão redimensionadas para **224×224 pixels** antes de serem alimentadas nos modelos, por ser o formato esperado pelas arquiteturas EfficientNetB0 e MobileNetV2, e também utilizado na CNN própria para padronização

#### Balanceamento das classes

O dataset apresenta um **leve desbalanceamento** entre as classes. A classe com mais amostras (*billiard_ball*) possui 808 imagens, enquanto a menor (*basketball*) possui 426 — uma razão de aproximadamente 1,9×. Esse nível de desbalanceamento é considerado moderado e não exige técnicas agressivas de reamostragem, mas será monitorado durante o treinamento por meio de métricas por classe (precisão, revocação e F1-score).


## 2.2 Análise das Classes e Dados


## 2.3 Pré-processamento


## 2.4 Arquitetura das Redes Neurais

Para o trabalho, foram consideradas duas abordagens complementares: uma CNN construída do zero, usada como linha de base, e redes pré-treinadas com *transfer learning*, especialmente EfficientNetB0 e MobileNetV2 (que são redes leves e eficientes e que uma delas será escolhida para o transfer learning).

### 2.4.1 CNN própria

A CNN própria foi pensada para ser simples, estável e fácil de interpretar. A arquitetura proposta segue a lógica de extração progressiva de características:

- **Camada de entrada:** imagens redimensionadas para um formato fixo, como 224 x 224 x 3.
- **Blocos convolucionais:** 3 blocos com convoluções 2D, ativação ReLU e *padding* igual.
- **Pooling:** *MaxPooling2D* após cada bloco convolucional para reduzir dimensionalidade e manter as informações mais relevantes.
- **Regularização:** *Dropout* entre os blocos e antes da saída para reduzir *overfitting*.
- **Classificação final:** camadas densas com *softmax* na saída, uma neurônio por classe.

Uma configuração coerente para essa CNN é:

- Bloco 1: 32 filtros, convolução 3 x 3, ReLU, MaxPooling
- Bloco 2: 64 filtros, convolução 3 x 3, ReLU, MaxPooling
- Bloco 3: 128 filtros, convolução 3 x 3, ReLU, MaxPooling
- *Flatten* ou *GlobalAveragePooling2D*
- *Dense* final com *softmax*

Essa estrutura é suficiente para capturar padrões visuais básicos e serve como referência para comparar com as redes pré-treinadas.

### 2.4.2 Transfer learning

A rede de *transfer learning* escolhida para o experimento principal foi a **EfficientNetB0**, por apresentar bom equilíbrio entre desempenho e custo computacional. A **MobileNetV2** foi mantida como comparação leve, útil quando a prioridade é reduzir parâmetros e acelerar inferência.

A estratégia adotada para a EfficientNetB0 (experimento principal):

1. Carregar os pesos pré-treinados no ImageNet.
2. Congelar a base convolucional nas primeiras etapas.
3. Adicionar uma cabeça de classificação específica para as classes do conjunto de dados.
4. Se necessário, liberar parte das últimas camadas para *fine-tuning*.

### 2.4.3 Justificativa das escolhas

As escolhas arquiteturais foram feitas considerando o tamanho do conjunto de dados e o objetivo de classificação de imagens esportivas:

- A **CNN própria** funciona como baseline e permite avaliar o quanto o problema pode ser resolvido sem conhecimento prévio transferido.
- **EfficientNetB0** tende a oferecer melhor relação entre profundidade, eficiência e generalização, sendo uma boa candidata para maior acurácia.
- **MobileNetV2** é uma alternativa mais leve, com menor custo de processamento, útil para comparação e para cenários com limitação de recursos.
- O uso de **Dropout** e de *MaxPooling* ajuda a controlar o sobreajuste e a reduzir o tamanho das representações intermediárias.
- A ativação **ReLU** é adequada por ser simples, eficiente e amplamente usada em CNNs modernas.

### 2.4.4 Comparação das arquiteturas

| Arquitetura | Número de camadas | Convoluções | Pooling | Dropout | Função de ativação | Vantagem principal |
| --- | --- | --- | --- | --- | --- | --- |
| CNN própria | 3 blocos convolucionais + classificadores | 3 x 3 | MaxPooling2D | Sim | ReLU / Softmax | Baseline simples e interpretável |
| EfficientNetB0 | Backbone pré-treinado + cabeça densa | Convoluções otimizadas pela família EfficientNet | GlobalAveragePooling2D ou pooling implícito | Sim | Swish/ReLU + Softmax | Melhor equilíbrio entre desempenho e custo |
| MobileNetV2 | Backbone pré-treinado + cabeça densa | Convoluções separáveis | GlobalAveragePooling2D | Sim | ReLU6 + Softmax | Menor custo computacional |

### 2.4.5 Diagrama simplificado

```mermaid
flowchart LR
    A[Imagem de entrada\n224 x 224 x 3] --> B{Estratégia}
    B --> C[CNN própria\nConv 3x3 + ReLU\nMaxPooling + Dropout]
    B --> D[EfficientNetB0\nbase congelada + cabeça densa]
    B --> E[MobileNetV2\nbase congelada + cabeça leve]
    C --> F[Softmax\nclassificação]
    D --> F
    E --> F
```

Em resumo, a CNN própria fornece a linha de base do estudo, enquanto EfficientNetB0 e MobileNetV2 representam as melhores alternativas de *transfer learning* para comparar desempenho, robustez e custo de execução.